# XLM-RoBERTa Multi-Task Finetuning
## Sentiment Analysis + Topic Classification on NEU-ESC Dataset

**Model:** `xlm-roberta-base`

**Tasks:**
- Sentiment Analysis (4 classes): Neutral, Positive, Negative, Toxic
- Topic Classification (10 classes): Spam, News, Academic, Other, Service, Jobs, Personal, Social, Help, Club

**Features:**
- Multi-task learning với shared backbone
- Class weights để xử lý imbalanced data
- Gradient accumulation
- Model checkpoint saving

**Note:** XLM-RoBERTa không cần word segmentation như PhoBERT

## 1. Setup

In [1]:
!pip install transformers accelerate -q

# Create directories for Vertex AI Workbench
import os
os.makedirs('/home/jupyter/data', exist_ok=True)
os.makedirs('/home/jupyter/checkpoints', exist_ok=True)

## 2. Configuration

In [2]:
import os
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from tqdm.auto import tqdm

@dataclass
class Config:
    # Paths (Vertex AI Workbench)
    train_path: str = "/home/jupyter/data/train_set.csv"
    val_path: str = "/home/jupyter/data/val_set.csv"
    test_path: str = "/home/jupyter/data/test_set.csv"
    checkpoint_path: str = "/home/jupyter/checkpoints/xlmroberta_multitask_best.pt"
    
    # Model
    model_name: str = "xlm-roberta-base"
    max_len: int = 256
    dropout: float = 0.3
    
    # Classes
    num_sentiment_classes: int = 4
    num_topic_classes: int = 10
    sentiment_labels: tuple = ("Neutral", "Positive", "Negative", "Toxic")
    topic_labels: tuple = ("Spam", "News", "Academic", "Other", "Service", "Jobs", "Personal", "Social", "Help", "Club")
    
    # Training
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    epochs: int = 15
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    
    # Multi-task Loss Weights
    alpha: float = 0.6
    beta: float = 0.4
    
    # Regularization
    label_smoothing: float = 0.1
    early_stopping_patience: int = 5
    
    seed: int = 42


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


config = Config()
set_seed(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


## 3. Load Data

In [3]:
def load_data(config: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print("Loading data...")
    
    train_df = pd.read_csv(config.train_path)
    val_df = pd.read_csv(config.val_path)
    test_df = pd.read_csv(config.test_path)
    
    for df in [train_df, val_df, test_df]:
        df['sentiment'] = df['sentiment'].astype(int)
        df['classification'] = df['classification'].astype(int)
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    
    return train_df, val_df, test_df


def compute_class_weights(labels: np.ndarray, num_classes: int) -> torch.Tensor:
    class_counts = np.bincount(labels, minlength=num_classes)
    class_counts = np.maximum(class_counts, 1)
    total = len(labels)
    weights = total / (num_classes * class_counts)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


train_df, val_df, test_df = load_data(config)

sentiment_weights = compute_class_weights(
    train_df['sentiment'].values,
    config.num_sentiment_classes
)

topic_weights = compute_class_weights(
    train_df['classification'].values,
    config.num_topic_classes
)

Loading data...


Train: 23048, Val: 3305, Test: 6613


## 4. Dataset

In [5]:
class NEUESCDataset(Dataset):
    def __init__(self, texts: List[str], sentiments: List[int], topics: List[int], tokenizer, max_len: int):
        self.texts = texts
        self.sentiments = sentiments
        self.topics = topics
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        sentiment = self.sentiments[idx]
        topic = self.topics[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(sentiment, dtype=torch.long),
            'topic_label': torch.tensor(topic, dtype=torch.long)
        }


def create_data_loaders(train_df: pd.DataFrame, val_df: pd.DataFrame, tokenizer, config: Config) -> Tuple[DataLoader, DataLoader]:
    train_dataset = NEUESCDataset(
        texts=train_df['text'].tolist(),
        sentiments=train_df['sentiment'].tolist(),
        topics=train_df['classification'].tolist(),
        tokenizer=tokenizer,
        max_len=config.max_len
    )
    
    val_dataset = NEUESCDataset(
        texts=val_df['text'].tolist(),
        sentiments=val_df['sentiment'].tolist(),
        topics=val_df['classification'].tolist(),
        tokenizer=tokenizer,
        max_len=config.max_len
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader

## 5. Model

In [6]:
class XLMRobertaMultiTask(nn.Module):
    def __init__(self, model_name: str, num_sentiment_classes: int = 4, num_topic_classes: int = 10, dropout: float = 0.1):
        super(XLMRobertaMultiTask, self).__init__()
        
        self.xlm_roberta = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.xlm_roberta.config.hidden_size
        
        self.shared_linear = nn.Linear(self.hidden_size, self.hidden_size)
        self.layer_norm = nn.LayerNorm(self.hidden_size)
        
        self.sentiment_classifier = nn.Linear(self.hidden_size, num_sentiment_classes)
        self.topic_classifier = nn.Linear(self.hidden_size, num_topic_classes)
        
        self._init_weights(self.shared_linear)
        self._init_weights(self.sentiment_classifier)
        self._init_weights(self.topic_classifier)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()
    
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        cls_output = self.shared_linear(cls_output)
        cls_output = self.layer_norm(cls_output)
        
        sentiment_logits = self.sentiment_classifier(cls_output)
        topic_logits = self.topic_classifier(cls_output)
        
        return sentiment_logits, topic_logits

## 6. Loss Functions

In [7]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, weight: torch.Tensor = None, gamma: float = 2.0, label_smoothing: float = 0.0, reduction: str = 'mean'):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none', label_smoothing=self.label_smoothing)
        p = F.softmax(inputs, dim=-1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class MultiTaskLoss(nn.Module):
    def __init__(self, sentiment_weights: torch.Tensor, topic_weights: torch.Tensor, alpha: float = 0.6, beta: float = 0.4, label_smoothing: float = 0.0, gamma: float = 2.0, device: torch.device = None):
        super(MultiTaskLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        
        sentiment_weights_device = sentiment_weights.to(device)
        topic_weights_device = topic_weights.to(device)
        
        self.sentiment_loss_fn = FocalLoss(weight=sentiment_weights_device, gamma=gamma, label_smoothing=label_smoothing)
        self.topic_loss_fn = FocalLoss(weight=topic_weights_device, gamma=gamma, label_smoothing=label_smoothing)
    
    def forward(self, sentiment_logits: torch.Tensor, topic_logits: torch.Tensor, sentiment_labels: torch.Tensor, topic_labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sentiment_loss = self.sentiment_loss_fn(sentiment_logits, sentiment_labels)
        topic_loss = self.topic_loss_fn(topic_logits, topic_labels)
        total_loss = self.alpha * sentiment_loss + self.beta * topic_loss
        return total_loss, sentiment_loss, topic_loss

## 7. Training Functions

In [8]:
def train_epoch(model: nn.Module, dataloader: DataLoader, loss_fn: MultiTaskLoss, optimizer: torch.optim.Optimizer, scheduler, device: torch.device, config: Config) -> Dict[str, float]:
    model.train()
    total_loss, total_sentiment_loss, total_topic_loss = 0.0, 0.0, 0.0
    num_batches = 0
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_labels = batch['sentiment_label'].to(device)
        topic_labels = batch['topic_label'].to(device)
        
        sentiment_logits, topic_logits = model(input_ids, attention_mask)
        loss, sent_loss, topic_loss = loss_fn(sentiment_logits, topic_logits, sentiment_labels, topic_labels)
        loss = loss / config.gradient_accumulation_steps
        loss.backward()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        total_sentiment_loss += sent_loss.item()
        total_topic_loss += topic_loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        progress_bar.set_postfix({'loss': f"{total_loss / num_batches:.4f}"})
    
    if (batch_idx + 1) % config.gradient_accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    return {'loss': total_loss / num_batches, 'sentiment_loss': total_sentiment_loss / num_batches, 'topic_loss': total_topic_loss / num_batches}


@torch.no_grad()
def eval_epoch(model: nn.Module, dataloader: DataLoader, loss_fn: MultiTaskLoss, device: torch.device) -> Dict[str, any]:
    model.eval()
    total_loss, total_sentiment_loss, total_topic_loss = 0.0, 0.0, 0.0
    num_batches = 0
    
    all_sentiment_preds, all_sentiment_labels = [], []
    all_topic_preds, all_topic_labels = [], []
    
    progress_bar = tqdm(dataloader, desc="Evaluating", leave=False)
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_labels = batch['sentiment_label'].to(device)
        topic_labels = batch['topic_label'].to(device)
        
        sentiment_logits, topic_logits = model(input_ids, attention_mask)
        loss, sent_loss, topic_loss = loss_fn(sentiment_logits, topic_logits, sentiment_labels, topic_labels)
        
        total_loss += loss.item()
        total_sentiment_loss += sent_loss.item()
        total_topic_loss += topic_loss.item()
        num_batches += 1
        
        sentiment_preds = torch.argmax(sentiment_logits, dim=1)
        topic_preds = torch.argmax(topic_logits, dim=1)
        
        all_sentiment_preds.extend(sentiment_preds.cpu().numpy())
        all_sentiment_labels.extend(sentiment_labels.cpu().numpy())
        all_topic_preds.extend(topic_preds.cpu().numpy())
        all_topic_labels.extend(topic_labels.cpu().numpy())
    
    return {
        'loss': total_loss / num_batches,
        'sentiment_loss': total_sentiment_loss / num_batches,
        'topic_loss': total_topic_loss / num_batches,
        'sentiment_preds': np.array(all_sentiment_preds),
        'sentiment_labels': np.array(all_sentiment_labels),
        'topic_preds': np.array(all_topic_preds),
        'topic_labels': np.array(all_topic_labels)
    }

## 8. Metrics

In [9]:
def compute_metrics(sentiment_preds: np.ndarray, sentiment_labels: np.ndarray, topic_preds: np.ndarray, topic_labels: np.ndarray, config: Config, print_report: bool = False) -> Dict[str, float]:
    sentiment_acc = accuracy_score(sentiment_labels, sentiment_preds)
    sentiment_f1 = f1_score(sentiment_labels, sentiment_preds, average='macro', zero_division=0)
    topic_acc = accuracy_score(topic_labels, topic_preds)
    topic_f1 = f1_score(topic_labels, topic_preds, average='macro', zero_division=0)
    combined_f1 = (sentiment_f1 + topic_f1) / 2
    
    metrics = {
        'sentiment_acc': sentiment_acc,
        'sentiment_f1': sentiment_f1,
        'topic_acc': topic_acc,
        'topic_f1': topic_f1,
        'combined_f1': combined_f1
    }
    
    if print_report:
        print("\n" + "="*60)
        print("SENTIMENT CLASSIFICATION REPORT")
        print("="*60)
        print(classification_report(sentiment_labels, sentiment_preds, labels=list(range(config.num_sentiment_classes)), target_names=list(config.sentiment_labels), zero_division=0))
        print("\n" + "="*60)
        print("TOPIC CLASSIFICATION REPORT")
        print("="*60)
        print(classification_report(topic_labels, topic_preds, labels=list(range(config.num_topic_classes)), target_names=list(config.topic_labels), zero_division=0))
    
    return metrics

## 9. Training Loop

In [10]:
def train_model(config: Config):
    print("\n" + "="*60)
    print("INITIALIZING")
    print("="*60)
    
    print(f"\nLoading tokenizer: {config.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    
    print("\nCreating data loaders...")
    train_loader, val_loader = create_data_loaders(train_df, val_df, tokenizer, config)
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    
    print(f"\nLoading model: {config.model_name}")
    model = XLMRobertaMultiTask(
        model_name=config.model_name,
        num_sentiment_classes=config.num_sentiment_classes,
        num_topic_classes=config.num_topic_classes,
        dropout=config.dropout
    )
    model = model.to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
    print("\nSetting up loss function...")
    loss_fn = MultiTaskLoss(
        sentiment_weights=sentiment_weights,
        topic_weights=topic_weights,
        alpha=config.alpha,
        beta=config.beta,
        label_smoothing=config.label_smoothing,
        device=device
    )
    
    print("\nSetting up optimizer with discriminative LR...")
    backbone_lr = 2e-5
    head_lr = 5e-5
    
    optimizer_grouped_parameters = [
        {'params': model.xlm_roberta.parameters(), 'lr': backbone_lr, 'weight_decay': config.weight_decay},
        {'params': model.shared_linear.parameters(), 'lr': head_lr, 'weight_decay': config.weight_decay},
        {'params': model.layer_norm.parameters(), 'lr': head_lr, 'weight_decay': 0.0},
        {'params': model.sentiment_classifier.parameters(), 'lr': head_lr, 'weight_decay': config.weight_decay},
        {'params': model.topic_classifier.parameters(), 'lr': head_lr, 'weight_decay': config.weight_decay}
    ]
    
    optimizer = AdamW(optimizer_grouped_parameters)
    print(f"  Backbone LR: {backbone_lr}")
    print(f"  Head LR: {head_lr}")
    
    total_steps = len(train_loader) * config.epochs // config.gradient_accumulation_steps
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    
    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)
    
    best_combined_f1 = 0.0
    epochs_without_improvement = 0
    history = {'train_loss': [], 'val_loss': [], 'sentiment_f1': [], 'topic_f1': [], 'combined_f1': []}
    
    for epoch in range(config.epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{config.epochs}")
        print(f"{'='*60}")
        
        train_results = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, device, config)
        print(f"\nTrain Loss: {train_results['loss']:.4f}")
        
        val_results = eval_epoch(model, val_loader, loss_fn, device)
        print(f"Val Loss: {val_results['loss']:.4f}")
        
        metrics = compute_metrics(val_results['sentiment_preds'], val_results['sentiment_labels'], val_results['topic_preds'], val_results['topic_labels'], config)
        print(f"\nMetrics:")
        print(f"  Sentiment - Acc: {metrics['sentiment_acc']:.4f}, F1: {metrics['sentiment_f1']:.4f}")
        print(f"  Topic     - Acc: {metrics['topic_acc']:.4f}, F1: {metrics['topic_f1']:.4f}")
        print(f"  Combined F1: {metrics['combined_f1']:.4f}")
        
        history['train_loss'].append(train_results['loss'])
        history['val_loss'].append(val_results['loss'])
        history['sentiment_f1'].append(metrics['sentiment_f1'])
        history['topic_f1'].append(metrics['topic_f1'])
        history['combined_f1'].append(metrics['combined_f1'])
        
        if metrics['combined_f1'] > best_combined_f1:
            best_combined_f1 = metrics['combined_f1']
            epochs_without_improvement = 0
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_combined_f1': best_combined_f1,
                'metrics': metrics,
                'config': config.__dict__
            }
            torch.save(checkpoint, config.checkpoint_path)
            print(f"\n  >> New best model saved! (Combined F1: {best_combined_f1:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"\n  >> No improvement for {epochs_without_improvement} epoch(s)")
            if epochs_without_improvement >= config.early_stopping_patience:
                print(f"\n*** Early stopping triggered! ***")
                break
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE")
    print("="*60)
    print(f"Best Combined F1: {best_combined_f1:.4f}")
    print(f"Model saved to: {config.checkpoint_path}")
    
    return model, history

## 10. Run Training

In [11]:
model, history = train_model(config)


INITIALIZING

Loading tokenizer: xlm-roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Val Loss: 0.4041

Metrics:
  Sentiment - Acc: 0.8221, F1: 0.7432
  Topic     - Acc: 0.7758, F1: 0.6369
  Combined F1: 0.6901

  >> No improvement for 3 epoch(s)

Epoch 15/15


Training:   0%|          | 0/1441 [00:00<?, ?it/s]


Train Loss: 0.0157


Evaluating:   0%|          | 0/207 [00:00<?, ?it/s]

Val Loss: 0.3995

Metrics:
  Sentiment - Acc: 0.8212, F1: 0.7413
  Topic     - Acc: 0.7764, F1: 0.6372
  Combined F1: 0.6892

  >> No improvement for 4 epoch(s)

TRAINING COMPLETE
Best Combined F1: 0.6925
Model saved to: /home/jupyter/checkpoints/xlmroberta_multitask_best.pt


## 11. Load Checkpoint & Inference

In [12]:
def load_checkpoint(checkpoint_path: str, config: Config, device: torch.device):
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model = XLMRobertaMultiTask(
        model_name=config.model_name,
        num_sentiment_classes=config.num_sentiment_classes,
        num_topic_classes=config.num_topic_classes,
        dropout=config.dropout
    )
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"  Loaded from epoch: {checkpoint['epoch']}")
    print(f"  Best Combined F1: {checkpoint['best_combined_f1']:.4f}")
    
    return model, checkpoint


def predict_single(model: nn.Module, text: str, tokenizer, config: Config, device: torch.device) -> Dict[str, str]:
    model.eval()
    
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=config.max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        sentiment_logits, topic_logits = model(input_ids, attention_mask)
    
    sentiment_pred = torch.argmax(sentiment_logits, dim=1).item()
    topic_pred = torch.argmax(topic_logits, dim=1).item()
    
    return {
        'text': text,
        'sentiment': config.sentiment_labels[sentiment_pred],
        'topic': config.topic_labels[topic_pred]
    }


print("\n" + "="*60)
print("TESTING SAVED MODEL")
print("="*60)

loaded_model, checkpoint = load_checkpoint(config.checkpoint_path, config, device)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

test_texts = [
    "Truong dai hoc nay day rat tot, thay co nhiet tinh",
    "Dich vu qua te, khong bao gio quay lai",
    "Tuyen dung nhan vien part-time luong cao"
]

print("\nSample predictions:")
print("-"*60)
for text in test_texts:
    result = predict_single(loaded_model, text, tokenizer, config, device)
    print(f"Text: {result['text'][:50]}...")
    print(f"  Sentiment: {result['sentiment']}")
    print(f"  Topic: {result['topic']}")
    print()


TESTING SAVED MODEL
Loading checkpoint from: /home/jupyter/checkpoints/xlmroberta_multitask_best.pt


  Loaded from epoch: 11
  Best Combined F1: 0.6925



Sample predictions:
------------------------------------------------------------
Text: Truong dai hoc nay day rat tot, thay co nhiet tinh...
  Sentiment: Toxic
  Topic: Academic

Text: Dich vu qua te, khong bao gio quay lai...
  Sentiment: Toxic
  Topic: Other

Text: Tuyen dung nhan vien part-time luong cao...
  Sentiment: Neutral
  Topic: Jobs



## 12. Evaluate on Test Set

In [13]:
print("="*60)
print("EVALUATION ON TEST SET")
print("="*60)

ckpt = torch.load(config.checkpoint_path, map_location=device)
model = XLMRobertaMultiTask(config.model_name, config.num_sentiment_classes, config.num_topic_classes, config.dropout).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model with Val F1: {ckpt['best_combined_f1']:.4f}")

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
test_ds = NEUESCDataset(test_df['text'].tolist(), test_df['sentiment'].tolist(), test_df['classification'].tolist(), tokenizer, config.max_len)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, num_workers=2)

task_loss = MultiTaskLoss(sentiment_weights, topic_weights, config.alpha, config.beta, config.label_smoothing, gamma=2.0, device=device)
test_results = eval_epoch(model, test_loader, task_loss, device)

test_metrics = compute_metrics(test_results['sentiment_preds'], test_results['sentiment_labels'], test_results['topic_preds'], test_results['topic_labels'], config, print_report=True)

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)
print(f"Sentiment - Accuracy: {test_metrics['sentiment_acc']:.4f}, Macro F1: {test_metrics['sentiment_f1']:.4f}")
print(f"Topic     - Accuracy: {test_metrics['topic_acc']:.4f}, Macro F1: {test_metrics['topic_f1']:.4f}")
print(f"Combined Macro F1: {test_metrics['combined_f1']:.4f}")

EVALUATION ON TEST SET


Loaded best model with Val F1: 0.6925


Evaluating:   0%|          | 0/414 [00:00<?, ?it/s]


SENTIMENT CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Neutral       0.89      0.83      0.86      4558
    Positive       0.60      0.73      0.66       834
    Negative       0.62      0.69      0.65      1046
       Toxic       0.80      0.88      0.84       175

    accuracy                           0.80      6613
   macro avg       0.73      0.78      0.75      6613
weighted avg       0.81      0.80      0.80      6613


TOPIC CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Spam       0.59      0.49      0.54        84
        News       0.46      0.52      0.49       183
    Academic       0.82      0.84      0.83      2104
       Other       0.86      0.83      0.85      2882
     Service       0.61      0.66      0.64       474
        Jobs       0.60      0.64      0.62       163
    Personal       0.49      0.49      0.49       298
      Social       0.51      0.44      0.47       155
        Help    